# smSoD on Colab

Runtime -> Change runtime type -> T4 GPU, then run the cells in order.

Colab deletes the files when the runtime stops, and it stops if the tab is closed or idle
for a long time. For the long benchmark set `USE_DRIVE = True`, the csv is then saved in
Google Drive after every run, and running the benchmark cell again continues from there.

In [ ]:
# benchmark settings
M = "50 100 200"
K = "2 4 8"
RUNS = 5
MAX_GEN = 2000
BACKENDS = "numpy cupy"
USE_DRIVE = False    # True = save results to Google Drive (MyDrive/smSoD)

In [ ]:
# where to save the results
import os
OUT = "/content/results"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/smSoD"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

In [ ]:
# install
os.chdir("/content")
if not os.path.exists("smSoD"):
    !git clone -q https://github.com/rayliangg/smSoD.git
os.chdir("/content/smSoD")
!git pull -q

# fixed versions, cupy-cuda12x + nvrtc from pip
!pip uninstall -y -q cupy cupy-cuda12x cupy-cuda11x > /tmp/pip.log 2>&1
!pip install -q numpy==2.3.3 cupy-cuda12x==13.6.0 fastrlock==0.8.3 nvidia-cuda-nvrtc-cu12==12.9.86 >> /tmp/pip.log 2>&1 && echo "install ok" || tail -20 /tmp/pip.log

nvrtc = !python -c "import nvidia.cuda_nvrtc as m, os; print(os.path.join(list(m.__path__)[0], 'lib'))"
os.environ["LD_LIBRARY_PATH"] = nvrtc[-1] + ":" + os.environ.get("LD_LIBRARY_PATH", "")

In [ ]:
# check gpu and cupy
!nvidia-smi -L
!python -c "import cupy as cp; c = cp.zeros(4, dtype=cp.int32); cp.add.at(c, cp.asarray([0, 1, 1, 3]), 1); print('cupy', cp.__version__, 'nvrtc', cp.cuda.nvrtc.getVersion(), c.get())"

In [ ]:
# quick test, 200 generations
!python smSoD_numpy.py --m 50 --k 2 --max_gen 200 --seed 1 --output_file ''
!python smSoD_cupy.py  --m 50 --k 2 --max_gen 200 --seed 1 --output_file ''

In [ ]:
# benchmark, results are saved after every run
!python benchmark.py --m {M} --k {K} --runs {RUNS} --max_gen {MAX_GEN} --backends {BACKENDS} --csv {OUT}/benchmark.csv

In [ ]:
# result table
import shutil
from IPython.display import Markdown
shutil.copy("README.md", OUT + "/README.md")
text = open("README.md").read()
Markdown(text[text.index("## Performance"):])

In [ ]:
# download the results (not needed with USE_DRIVE)
from google.colab import files
files.download(OUT + "/benchmark.csv")
files.download(OUT + "/README.md")